In [1]:
import pandas as pd

In [2]:
players = pd.read_csv(r"C:\Users\saarp\Documents\GitHub\portfolio\Projects\Fantasy\Kaggle\Player Per Game.csv")

## The Stable players funnel

In [6]:
players.columns

Index(['season', 'lg', 'player', 'player_id', 'age', 'team', 'pos', 'g', 'gs',
       'mp_per_game', 'fg_per_game', 'fga_per_game', 'fg_percent',
       'x3p_per_game', 'x3pa_per_game', 'x3p_percent', 'x2p_per_game',
       'x2pa_per_game', 'x2p_percent', 'e_fg_percent', 'ft_per_game',
       'fta_per_game', 'ft_percent', 'orb_per_game', 'drb_per_game',
       'trb_per_game', 'ast_per_game', 'stl_per_game', 'blk_per_game',
       'tov_per_game', 'pf_per_game', 'pts_per_game'],
      dtype='object')

In [75]:
pipe_one = players[(players['season']>2023) &
             (players['age']<=32)]


players_info = ['player','season','age', 'team', 'pos']
parameters_to_count = ['g','mp_per_game','pts_per_game',
                       'tov_per_game',
                       'trb_per_game', 'ast_per_game', 'stl_per_game', 'blk_per_game',
                       'fta_per_game', 'ft_percent',
                    'x3pa_per_game', 'x3p_percent']

clear_pipe_one = pipe_one[players_info + parameters_to_count].sort_values(by=['player','season']).copy()
means_df = clear_pipe_one.groupby(by='player', as_index=False)[parameters_to_count].mean().round(2)

In [76]:
clear_pipe_one[clear_pipe_one['season']==2025]

clear_pipe_one['minplayed2025_class'] = 'D'
clear_pipe_one.loc[(clear_pipe_one['season']==2025) & 
                   (clear_pipe_one['mp_per_game']>10), 'minplayed2025_class'] = 'C'
clear_pipe_one.loc[(clear_pipe_one['season']==2025) & 
                   (clear_pipe_one['mp_per_game']>15), 'minplayed2025_class'] = 'B'
clear_pipe_one.loc[(clear_pipe_one['season']==2025) & 
                   (clear_pipe_one['mp_per_game']>20), 'minplayed2025_class'] = 'A'

#### Filtering players:

- minuts playing classes: (above 20 = A , above 15 = B , above 10 = C) relevant for 2025 season only

In [94]:
class_df = clear_pipe_one.loc[clear_pipe_one['season']==2025, ['player','minplayed2025_class','pos']]
class_df = class_df.drop_duplicates(subset='player')
merged_df = means_df.merge(class_df, on='player', how='left')

merged_df = merged_df[merged_df['minplayed2025_class'].isin(['A','B'])]

- only 65+ games per season:

In [95]:
f1 = merged_df[merged_df['g']>=60]
f1.head(5)

,player,g,mp_per_game,pts_per_game,tov_per_game,trb_per_game,ast_per_game,stl_per_game,blk_per_game,fta_per_game,ft_percent,x3pa_per_game,x3p_percent,minplayed2025_class,pos
0,A.J. Green,64.5,16.85,5.95,0.35,1.75,1.00,0.35,0.10,0.35,0.86,4.00,0.42,A,SG
4,Aaron Gordon,62.0,29.95,14.30,1.40,5.65,3.35,0.65,0.45,3.60,0.73,2.65,0.36,A,PF
7,Aaron Wiggins,77.0,19.30,9.45,0.80,3.15,1.45,0.75,0.20,1.05,0.81,3.05,0.44,A,SG
15,Alex Caruso,62.5,24.00,8.60,1.05,3.35,3.00,1.65,0.80,1.15,0.79,3.90,0.38,B,SG
20,Alex Sarr,67.0,27.10,13.00,1.70,6.50,2.40,0.70,1.50,2.50,0.68,5.10,0.31,A,C


- "3rd"-"10th" standing spot teams: as the highest potential for long and competative season run:

In [96]:
second_pipe = ['NYK', 'HOU', 'MIA','MEM', 'MIN', 'DEN','GSW', 'MIL', 'ORL','LAL','DAL','IND','DET','LAC','POR']
filter1 = pipe_one[(pipe_one['team'].isin(second_pipe)) &
                    (pipe_one['season'] == 2025)]['player'].unique().tolist()

f2 = f1[f1['player'].isin(filter1)].copy()


In [97]:
var_to_sort_by = 'pts_per_game'
sec_var_to_sort_by = 'fta_per_game'
pos = 'C'
pd.set_option('display.max_rows', 500)

f2[f2['pos']== pos].sort_values(
                            by=['minplayed2025_class', var_to_sort_by ,sec_var_to_sort_by],
                            ascending=[True, False, False])

,player,g,mp_per_game,pts_per_game,tov_per_game,trb_per_game,ast_per_game,stl_per_game,blk_per_game,fta_per_game,ft_percent,x3pa_per_game,x3p_percent,minplayed2025_class,pos
480,Nikola Jokić,74.5,35.65,28.00,3.15,12.55,9.60,1.60,0.75,5.95,0.81,3.80,0.39,A,C
359,Karl-Anthony Towns,67.0,33.85,23.10,2.80,10.55,3.05,0.85,0.70,5.20,0.85,5.00,0.42,A,C
283,Jaren Jackson Jr.,70.0,31.00,22.35,2.25,5.55,2.15,1.20,1.55,5.85,0.79,5.40,0.35,A,C
22,Alperen Şengün,69.5,32.00,20.10,2.60,9.80,4.95,1.15,0.75,5.60,0.69,1.50,0.26,A,C
45,Bam Adebayo,74.5,34.15,18.70,2.20,10.00,4.10,1.20,0.80,4.85,0.76,1.70,0.36,A,C
466,Myles Turner,74.5,28.60,16.35,1.55,6.70,1.40,0.65,1.95,3.70,0.77,4.85,0.38,A,C
234,Ivica Zubac,74.0,29.60,14.25,1.40,10.90,2.05,0.50,1.15,2.70,0.69,0.00,NaN,A,C
474,Naz Reid,80.5,25.85,13.85,1.40,5.60,1.80,0.75,0.90,1.80,0.76,5.40,0.40,A,C
535,Rudy Gobert,74.0,33.65,13.00,1.40,11.90,1.55,0.75,1.75,4.45,0.66,0.00,0.00,A,C
261,Jalen Duren,69.5,27.60,12.80,1.90,10.95,2.55,0.60,0.95,3.05,0.73,0.05,0.00,A,C
